# Agentic AI + Supabase Training Lab

Flow หลัก:

**Excel → Supabase PostgreSQL → PDF → Chunk → Embedding → pgvector → Semantic Search → RAG**

ใช้ Dataset บริษัทจำลอง ABC Beverage Co., Ltd.

## STEP 1 — Install Packages

In [ ]:
%pip install -q openai google-genai supabase pandas openpyxl pypdf python-dotenv requests


## STEP 2 — Import Libraries

In [ ]:
from pathlib import Path
from typing import Any
import os
import re
import json
import requests
import pandas as pd

from dotenv import load_dotenv
from supabase import create_client
from openai import OpenAI
from google import genai
from pypdf import PdfReader


## STEP 3 — Configure Supabase + OpenAI

สร้างไฟล์ `.env` ในโฟลเดอร์เดียวกับ Notebook:

```env
SUPABASE_URL=https://YOUR_PROJECT.supabase.co
SUPABASE_KEY=YOUR_SERVER_SIDE_KEY
OPENAI_API_KEY=YOUR_OPENAI_API_KEY
SUPABASE_STORAGE_BUCKET=company-documents
EMBEDDING_MODEL=text-embedding-3-small
```

In [ ]:
# ============================================================
# STEP 3 — FULL CONFIGURATION
# Supabase + Typhoon LLM + Gemini Embedding
# ============================================================

import os
from dotenv import load_dotenv
from supabase import create_client
from openai import OpenAI
from google import genai

load_dotenv(override=True)

# -------------------------
# Supabase
# -------------------------
SUPABASE_URL = os.getenv("SUPABASE_URL", "").strip()

SUPABASE_KEY = (
    os.getenv("SUPABASE_SECRET_KEY", "").strip()
    or os.getenv("SUPABASE_SERVICE_ROLE_KEY", "").strip()
    or os.getenv("SUPABASE_KEY", "").strip()
)

# -------------------------
# AI API Keys
# -------------------------
TYPHOON_API_KEY = os.getenv("TYPHOON_API_KEY", "").strip()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "").strip()

# -------------------------
# Models
# -------------------------
TYPHOON_MODEL = os.getenv(
    "TYPHOON_MODEL",
    "typhoon-v2.5-30b-a3b-instruct"
).strip()

GEMINI_EMBEDDING_MODEL = os.getenv(
    "GEMINI_EMBEDDING_MODEL",
    "gemini-embedding-001"
).strip()

EMBEDDING_DIMENSIONS = int(
    os.getenv("EMBEDDING_DIMENSIONS", "1536")
)

# -------------------------
# Storage
# -------------------------
STORAGE_BUCKET = os.getenv(
    "SUPABASE_STORAGE_BUCKET",
    "company-documents"
).strip()

# -------------------------
# Optional External Open API
# -------------------------
OPEN_API_BASE_URL = os.getenv(
    "OPEN_API_BASE_URL",
    ""
).strip().rstrip("/")

OPEN_API_TOKEN = os.getenv(
    "OPEN_API_TOKEN",
    ""
).strip()

# -------------------------
# Validation
# -------------------------
assert SUPABASE_URL, "Missing SUPABASE_URL"
assert SUPABASE_KEY, "Missing SUPABASE_SECRET_KEY / SUPABASE_SERVICE_ROLE_KEY"
assert TYPHOON_API_KEY, "Missing TYPHOON_API_KEY"
assert GEMINI_API_KEY, "Missing GEMINI_API_KEY"

# -------------------------
# Clients
# -------------------------
supabase = create_client(
    SUPABASE_URL,
    SUPABASE_KEY
)

# Typhoon API is OpenAI-compatible
typhoon_client = OpenAI(
    api_key=TYPHOON_API_KEY,
    base_url="https://api.opentyphoon.ai/v1"
)

# Gemini client is used for embeddings
gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("=" * 60)
print("Configuration loaded")
print("=" * 60)
print("LLM       : Typhoon")
print("Model     :", TYPHOON_MODEL)
print("Embedding : Gemini")
print("Embed model:", GEMINI_EMBEDDING_MODEL)
print("Vector dim:", EMBEDDING_DIMENSIONS)
print("Supabase  : READY")
print("=" * 60)


### `.env` สำหรับ Lab นี้

ใช้ **Typhoon เป็น LLM/Agent** และ **Gemini เป็น Embedding**

```env
SUPABASE_URL=https://YOUR_PROJECT.supabase.co
SUPABASE_SECRET_KEY=sb_secret_xxxxx

TYPHOON_API_KEY=xxxxx
GEMINI_API_KEY=xxxxx

TYPHOON_MODEL=typhoon-v2.5-30b-a3b-instruct
GEMINI_EMBEDDING_MODEL=gemini-embedding-001
EMBEDDING_DIMENSIONS=1536

SUPABASE_STORAGE_BUCKET=company-documents
```

ไม่ต้องมี `OPENAI_API_KEY` สำหรับเวอร์ชันนี้


## STEP 4 — Prepare Supabase

ก่อนรัน Notebook ให้เปิด **Supabase → SQL Editor** แล้วรันไฟล์ `setup_supabase.sql`

ไฟล์นี้จะสร้าง:
- customers
- products
- inventory
- invoices
- payments
- employees
- knowledge_documents
- pgvector extension
- `match_documents()` RPC

## STEP 5 — Locate Dataset Files

In [ ]:
BASE_DIR = Path(".")

EXCEL_PATH = BASE_DIR / "ABC_Beverage_Supabase_Ready.xlsx"
PDF_DIR = BASE_DIR / "ABC_Beverage_RAG_Documents"

print("Excel:", EXCEL_PATH.exists())
print("PDF folder:", PDF_DIR.exists())

if PDF_DIR.exists():
    for p in sorted(PDF_DIR.glob("*.pdf")):
        print("-", p.name)

## STEP 6 — Helper Functions สำหรับ Excel

In [ ]:
SHEET_CONFIG = {
    "Customers": ("customers", "customer_code"),
    "Products": ("products", "sku"),
    "Inventory": ("inventory", "sku,warehouse"),
    "Invoices": ("invoices", "invoice_no"),
    "Payments": ("payments", "payment_no"),
    "Employees": ("employees", "employee_code"),
}

def json_safe(value: Any) -> Any:
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    if isinstance(value, pd.Timestamp):
        if value.hour == 0 and value.minute == 0 and value.second == 0:
            return value.date().isoformat()
        return value.isoformat()

    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass

    return value

def dataframe_records(df):
    return [
        {str(k): json_safe(v) for k, v in row.items()}
        for row in df.to_dict(orient="records")
    ]

def batched(items, size=100):
    for i in range(0, len(items), size):
        yield items[i:i+size]

## STEP 7 — Preview Excel Data

In [ ]:
xls = pd.ExcelFile(EXCEL_PATH)
print(xls.sheet_names)

customers_preview = pd.read_excel(EXCEL_PATH, sheet_name="Customers")
customers_preview.head()

### STEP 8A — Validate Excel Columns Before Upload

ไฟล์ `ABC_Beverage_Supabase_Ready.xlsx` ถูกปรับให้ header อยู่แถว 1 และตรงกับ PostgreSQL schema

Cell นี้จะหยุดทันทีถ้าชื่อ column ไม่ตรง เพื่อให้รู้ปัญหาก่อนส่งไป Supabase


In [ ]:
EXPECTED_COLUMNS = {
    "Customers": [
        "customer_code", "customer_name", "customer_type", "province",
        "credit_term_days", "credit_limit_thb", "risk_level", "status"
    ],
    "Products": [
        "sku", "product_name", "category", "unit_price_thb",
        "minimum_stock", "cost_thb"
    ],
    "Inventory": [
        "sku", "warehouse", "quantity_on_hand",
        "minimum_stock", "stock_status"
    ],
    "Invoices": [
        "invoice_no", "customer_code", "customer_name",
        "invoice_date", "due_date", "amount_thb",
        "paid_amount_thb", "outstanding_thb",
        "status", "overdue_days"
    ],
    "Payments": [
        "payment_no", "invoice_no", "customer_code",
        "payment_date", "amount_thb", "payment_method"
    ],
    "Employees": [
        "employee_code", "employee_name", "department",
        "position", "salary_thb", "start_date", "status"
    ],
}

def validate_excel_schema(excel_path):
    problems = []

    for sheet_name, expected in EXPECTED_COLUMNS.items():
        df = pd.read_excel(excel_path, sheet_name=sheet_name)
        actual = list(df.columns)

        if actual != expected:
            problems.append({
                "sheet": sheet_name,
                "expected": expected,
                "actual": actual,
            })
        else:
            print(f"✅ {sheet_name}: {len(actual)} columns OK")

    if problems:
        print("\n❌ Schema mismatch found")
        for p in problems:
            print("\nSheet:", p["sheet"])
            print("Expected:", p["expected"])
            print("Actual:  ", p["actual"])
        raise ValueError("Excel columns do not match Supabase schema")

    print("\n✅ All Excel sheets match PostgreSQL schema")

validate_excel_schema(EXCEL_PATH)

## STEP 8 — Upload Excel → Supabase PostgreSQL

In [ ]:
def load_excel_to_supabase(excel_path, batch_size=100):
    validate_excel_schema(excel_path)

    for sheet_name, (table_name, conflict_columns) in SHEET_CONFIG.items():
        df = pd.read_excel(excel_path, sheet_name=sheet_name).dropna(how="all")

        # Normalize date columns to PostgreSQL-friendly YYYY-MM-DD strings
        date_columns = {
            "Invoices": ["invoice_date", "due_date"],
            "Payments": ["payment_date"],
            "Employees": ["start_date"],
        }

        for col in date_columns.get(sheet_name, []):
            df[col] = pd.to_datetime(df[col], errors="raise").dt.strftime("%Y-%m-%d")

        # Force integer fields to actual integers
        int_columns = {
            "Customers": ["credit_term_days"],
            "Products": ["minimum_stock"],
            "Inventory": ["quantity_on_hand", "minimum_stock"],
            "Invoices": ["overdue_days"],
        }

        for col in int_columns.get(sheet_name, []):
            df[col] = pd.to_numeric(df[col], errors="raise").astype(int)

        records = dataframe_records(df)

        print(f"{sheet_name} -> {table_name}: {len(records)} rows")

        for chunk in batched(records, batch_size):
            (
                supabase.table(table_name)
                .upsert(chunk, on_conflict=conflict_columns)
                .execute()
            )

    print("✅ Excel import complete")

In [ ]:
load_excel_to_supabase(EXCEL_PATH)

## STEP 9 — Test Structured Data

In [ ]:
result = (
    supabase.table("invoices")
    .select("*")
    .gt("outstanding_thb", 0)
    .order("overdue_days", desc=True)
    .limit(10)
    .execute()
)

pd.DataFrame(result.data)

## STEP 10 — PDF Text Extraction

In [ ]:
def normalize_text(text):
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def extract_pdf_text(pdf_path):
    reader = PdfReader(str(pdf_path))
    pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        page_text = normalize_text(page.extract_text() or "")
        if page_text:
            pages.append(f"[Page {page_number}] {page_text}")

    return "\n".join(pages)

In [ ]:
sample_pdf = sorted(PDF_DIR.glob("*.pdf"))[0]
text = extract_pdf_text(sample_pdf)

print(sample_pdf.name)
print(text[:1500])

## STEP 11 — Chunking

In [ ]:
def chunk_text(text, chunk_size=900, overlap=150):
    text = normalize_text(text)

    if not text:
        return []

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        if end < len(text):
            window = text[start:end]
            best = max(
                window.rfind("\n"),
                window.rfind("。"),
                window.rfind(". "),
                window.rfind(" ")
            )

            if best > chunk_size * 0.60:
                end = start + best + 1

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = max(end - overlap, start + 1)

    return chunks

In [ ]:
chunks = chunk_text(text)

print("Chunks:", len(chunks))
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i} ---")
    print(chunk[:800])

## STEP 12 — Create Embedding

In [ ]:
def embed_text(text):
    """
    Create a Gemini embedding with 1536 dimensions
    to match Supabase vector(1536).
    """

    response = gemini_client.models.embed_content(
        model=GEMINI_EMBEDDING_MODEL,
        contents=text,
        config={
            "output_dimensionality": EMBEDDING_DIMENSIONS
        }
    )

    vector = response.embeddings[0].values

    if len(vector) != EMBEDDING_DIMENSIONS:
        raise ValueError(
            f"Embedding dimension mismatch: "
            f"expected {EMBEDDING_DIMENSIONS}, got {len(vector)}"
        )

    return vector


In [ ]:
sample_embedding = embed_text(chunks[0])

print("Embedding dimensions:", len(sample_embedding))
print(sample_embedding[:10])

## STEP 13 — Create Supabase Storage Bucket

In [ ]:
try:
    supabase.storage.create_bucket(
        STORAGE_BUCKET,
        options={"public": False}
    )
    print("Created bucket:", STORAGE_BUCKET)
except Exception:
    print("Bucket may already exist:", STORAGE_BUCKET)

## STEP 14 — Upload PDF + Store Chunks in pgvector

In [ ]:
def upload_pdf_to_storage(pdf_path):
    storage_path = f"rag/{pdf_path.name}"

    with pdf_path.open("rb") as f:
        try:
            supabase.storage.from_(STORAGE_BUCKET).upload(
                path=storage_path,
                file=f,
                file_options={
                    "content-type": "application/pdf",
                    "upsert": "true"
                }
            )
        except Exception as e:
            print("Storage upload warning:", e)

    return storage_path

In [ ]:
def ingest_pdf(pdf_path, chunk_size=900, overlap=150):
    print("Processing:", pdf_path.name)

    storage_path = upload_pdf_to_storage(pdf_path)
    text = extract_pdf_text(pdf_path)
    chunks = chunk_text(text, chunk_size, overlap)

    (
        supabase.table("knowledge_documents")
        .delete()
        .eq("source_file", pdf_path.name)
        .execute()
    )

    rows = []

    for idx, chunk in enumerate(chunks):
        embedding = embed_text(chunk)

        rows.append({
            "title": pdf_path.stem.replace("_", " "),
            "source_file": pdf_path.name,
            "storage_path": storage_path,
            "chunk_index": idx,
            "content": chunk,
            "metadata": {
                "file_name": pdf_path.name,
                "chunk_index": idx,
                "embedding_model": EMBEDDING_MODEL
            },
            "embedding": embedding
        })

        print(f"  embedded {idx+1}/{len(chunks)}")

    for batch in batched(rows, 20):
        supabase.table("knowledge_documents").insert(batch).execute()

    print("Inserted chunks:", len(rows))

## STEP 15 — Ingest All PDFs

In [ ]:
for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
    ingest_pdf(pdf_path)

## STEP 16 — Check RAG Table

In [ ]:
result = (
    supabase.table("knowledge_documents")
    .select("id,title,source_file,chunk_index,content")
    .limit(20)
    .execute()
)

pd.DataFrame(result.data)

## STEP 17 — Semantic Search

In [ ]:
def semantic_search(question, match_count=5, match_threshold=0.30):
    query_embedding = embed_text(question)

    response = (
        supabase.rpc(
            "match_documents",
            {
                "query_embedding": query_embedding,
                "match_threshold": match_threshold,
                "match_count": match_count
            }
        )
        .execute()
    )

    return response.data or []

In [ ]:
question = "ถ้าลูกค้าค้างเกิน 60 วันต้องทำอะไร"

results = semantic_search(question)
pd.DataFrame(results)

## STEP 18 — Inspect Retrieved Context

In [ ]:
for rank, row in enumerate(results, start=1):
    print(
        f"#{rank} | similarity={row['similarity']:.4f} "
        f"| source={row['source_file']}"
    )
    print(row["content"])
    print("-" * 100)

## STEP 19 — Simple RAG Answer

LLM จะได้รับเฉพาะ Context ที่ดึงมาจาก Vector Search และถูกสั่งว่าอย่าแต่งข้อมูลเพิ่ม

### STEP 19A — Provider-Agnostic LLM Function

ฟังก์ชันนี้ทำให้ RAG สลับระหว่าง Gemini และ OpenAI ได้จาก `.env` โดยไม่ต้องแก้ logic ส่วนอื่น


In [ ]:
def ask_llm(system_prompt, user_prompt):
    """
    Typhoon LLM helper.
    """

    response = typhoon_client.chat.completions.create(
        model=TYPHOON_MODEL,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0.2,
        max_tokens=1200
    )

    return response.choices[0].message.content


In [ ]:
def rag_answer(question):
    results = semantic_search(question)

    if not results:
        return "ไม่พบข้อมูลที่เกี่ยวข้องใน Knowledge Base"

    context = "\n\n".join(
        [
            f"SOURCE: {r['source_file']}\n{r['content']}"
            for r in results
        ]
    )

    system_prompt = (
        "You are a company knowledge assistant. "
        "Answer only from the supplied context. "
        "If the answer is not in the context, "
        "say that the information was not found."
    )

    user_prompt = (
        f"QUESTION:\n{question}\n\n"
        f"CONTEXT:\n{context}"
    )

    return ask_llm(
        system_prompt=system_prompt,
        user_prompt=user_prompt
    )


In [ ]:
answer = rag_answer(
    "ถ้าลูกค้าค้างชำระเกิน 60 วัน บริษัทควรทำอย่างไร"
)

print(answer)

# STEP 20 — จาก RAG ไป Agentic AI

ตอนนี้เรามีข้อมูลสองประเภทแล้ว:

### SQL
`สินค้าไหน Stock ต่ำกว่า Minimum?`

→ Query PostgreSQL

### RAG
`ถ้าลูกค้าค้างเกิน 60 วัน Policy กำหนดไว้อย่างไร?`

→ Vector Search

### SQL + RAG
`ลูกค้ารายไหนควร Hold Order ตอนนี้?`

→ Agent ต้อง:
1. Query Invoice
2. ค้น Credit Policy
3. รวมผล
4. วิเคราะห์
5. Recommendation

Lab ถัดไปสามารถเพิ่ม Tool Calling ให้ Agent เลือก SQL หรือ RAG เองได้

# PART 2 — Agentic AI: Tool Calling + SQL + RAG + Open API + Action

จาก PART 1 เรามี PostgreSQL และ RAG แล้ว

PART 2 จะเพิ่ม Agent ที่สามารถ:

1. เลือก SQL Tool เมื่อคำถามเป็นข้อมูล Transaction
2. เลือก RAG Tool เมื่อคำถามเป็น Policy / SOP
3. เรียก SQL + RAG หลาย Tool ในคำถามเดียว
4. เรียก Open API ภายนอก (ถ้ามี)
5. สร้าง Action แบบ DRAFT และรอคนอนุมัติ
6. เก็บ Tool Trace เพื่อทดสอบว่า Agent เลือก Tool ถูกหรือไม่

Flow:

User → LLM Agent → Tool Call → Tool Result → LLM → Final Answer


## STEP 21 — Update Packages / Imports

`requests` ใช้สำหรับตัวอย่าง Open API ภายนอก


In [ ]:
%pip install -q requests


In [ ]:
import json
import requests
from datetime import datetime, timezone


## STEP 22 — Agent Configuration

เพิ่มใน `.env` ได้:

```env
AGENT_MODEL=gpt-5-mini

# Optional: ระบบ Open API ภายนอก
OPEN_API_BASE_URL=https://api.example.com
OPEN_API_TOKEN=YOUR_TOKEN
```

ถ้ายังไม่มี Open API จริง Agent จะยังใช้ SQL + RAG ได้ตามปกติ


In [ ]:
AGENT_MODEL = TYPHOON_MODEL

print("Agent LLM:", "Typhoon")
print("Agent model:", AGENT_MODEL)
print("External API configured:", bool(OPEN_API_BASE_URL))


## STEP 23 — SQL Tools

สำคัญ: ใน Lab นี้เรา **ไม่ให้ LLM เขียน SQL อิสระ**

เราสร้าง Tool ที่กำหนดขอบเขตไว้ชัดเจน เพื่อ:
- คุม Security
- ตรวจสอบง่าย
- ลด SQL hallucination


In [ ]:
def get_overdue_invoices(customer_name=None, min_overdue_days=1, limit=20):
    query = (
        supabase.table("invoices")
        .select(
            "invoice_no,customer_code,customer_name,invoice_date,due_date,"
            "amount_thb,paid_amount_thb,outstanding_thb,status,overdue_days"
        )
        .gt("outstanding_thb", 0)
        .gte("overdue_days", int(min_overdue_days))
        .order("overdue_days", desc=True)
        .limit(int(limit))
    )

    if customer_name:
        query = query.ilike("customer_name", f"%{customer_name}%")

    result = query.execute()
    return result.data or []


def get_low_stock(limit=50):
    result = (
        supabase.table("inventory")
        .select("sku,warehouse,quantity_on_hand,minimum_stock,stock_status")
        .eq("stock_status", "REORDER")
        .limit(int(limit))
        .execute()
    )
    return result.data or []


def get_customer(customer_name):
    result = (
        supabase.table("customers")
        .select(
            "customer_code,customer_name,customer_type,province,"
            "credit_term_days,credit_limit_thb,risk_level,status"
        )
        .ilike("customer_name", f"%{customer_name}%")
        .limit(10)
        .execute()
    )
    return result.data or []


## STEP 24 — RAG Tool

Tool นี้เรียก `semantic_search()` ที่เราสร้างไว้ใน PART 1


In [ ]:
def search_company_knowledge(question, match_count=5):
    results = semantic_search(
        question=question,
        match_count=int(match_count),
        match_threshold=0.30,
    )

    return [
        {
            "source_file": r.get("source_file"),
            "chunk_index": r.get("chunk_index"),
            "content": r.get("content"),
            "similarity": r.get("similarity"),
        }
        for r in results
    ]


## STEP 25 — Optional Open API Tool

ตัวอย่างนี้เป็น **Read-only GET Tool**

กำหนด Resource ที่อนุญาตไว้เท่านั้น:
- customers
- invoices
- inventory

ก่อนใช้กับ API จริง ให้แก้ path/parameter ให้ตรงกับ Swagger / OpenAPI Spec ของระบบคุณ


In [ ]:
ALLOWED_API_RESOURCES = {
    "customers",
    "invoices",
    "inventory",
}


def get_external_api(resource, identifier=None):
    if not OPEN_API_BASE_URL:
        return {
            "configured": False,
            "message": "OPEN_API_BASE_URL is not configured"
        }

    if resource not in ALLOWED_API_RESOURCES:
        return {
            "error": f"Resource '{resource}' is not allowed"
        }

    url = f"{OPEN_API_BASE_URL}/{resource}"
    if identifier:
        url += f"/{identifier}"

    headers = {"Accept": "application/json"}

    if OPEN_API_TOKEN:
        headers["Authorization"] = f"Bearer {OPEN_API_TOKEN}"

    response = requests.get(
        url,
        headers=headers,
        timeout=15,
    )

    response.raise_for_status()

    try:
        return response.json()
    except ValueError:
        return {"text": response.text}


## STEP 26 — Action Tool: Create Hold Order Draft

เพื่อความปลอดภัย Agent จะสร้างแค่ **DRAFT**

ยังไม่ Hold Order จริงจนกว่าจะมี Human Approval


In [ ]:
def create_hold_order_draft(
    customer_code,
    customer_name,
    reason,
    evidence,
):
    row = {
        "action_type": "HOLD_ORDER",
        "customer_code": customer_code,
        "customer_name": customer_name,
        "reason": reason,
        "evidence": evidence,
        "status": "DRAFT",
    }

    result = (
        supabase.table("agent_actions")
        .insert(row)
        .execute()
    )

    return result.data or [row]


## STEP 27 — Tool Definitions สำหรับ LLM

OpenAI Function Calling ใช้ JSON Schema เพื่อบอก Agent ว่า Tool แต่ละตัวทำอะไรและรับ Parameter อะไร


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_overdue_invoices",
            "description": (
                "Get structured invoice data from Supabase. "
                "Use for outstanding amount, due date, and overdue-day questions."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_name": {
                        "type": ["string", "null"],
                        "description": "Optional customer name filter"
                    },
                    "min_overdue_days": {
                        "type": "integer"
                    },
                    "limit": {
                        "type": "integer"
                    }
                },
                "required": [
                    "customer_name",
                    "min_overdue_days",
                    "limit"
                ],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_low_stock",
            "description": "Get inventory rows that require reorder.",
            "parameters": {
                "type": "object",
                "properties": {
                    "limit": {"type": "integer"}
                },
                "required": ["limit"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_customer",
            "description": "Get customer master data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_name": {"type": "string"}
                },
                "required": ["customer_name"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_company_knowledge",
            "description": (
                "Search company PDF knowledge using RAG. "
                "Use for policies, SOPs, rules and procedures."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {"type": "string"},
                    "match_count": {"type": "integer"}
                },
                "required": ["question", "match_count"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_external_api",
            "description": (
                "Read data from a configured external Open API."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "resource": {
                        "type": "string",
                        "enum": ["customers", "invoices", "inventory"]
                    },
                    "identifier": {
                        "type": ["string", "null"]
                    }
                },
                "required": ["resource", "identifier"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "create_hold_order_draft",
            "description": (
                "Create a DRAFT hold-order action. "
                "Human approval is required."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_code": {"type": "string"},
                    "customer_name": {"type": "string"},
                    "reason": {"type": "string"},
                    "evidence": {"type": "object"}
                },
                "required": [
                    "customer_code",
                    "customer_name",
                    "reason",
                    "evidence"
                ],
                "additionalProperties": False
            }
        }
    }
]


## STEP 28 — Tool Router

LLM เลือกชื่อ Tool ส่วน Python เป็นผู้ Execute Tool จริง


In [ ]:
TOOL_FUNCTIONS = {
    "get_overdue_invoices": get_overdue_invoices,
    "get_low_stock": get_low_stock,
    "get_customer": get_customer,
    "search_company_knowledge": search_company_knowledge,
    "get_external_api": get_external_api,
    "create_hold_order_draft": create_hold_order_draft,
}


def execute_tool(name, arguments):
    if name not in TOOL_FUNCTIONS:
        raise ValueError(f"Unknown tool: {name}")

    return TOOL_FUNCTIONS[name](**arguments)


## STEP 29 — Agent Loop

Agent Loop มีหน้าที่:

1. ส่งคำถาม + Tools ให้ LLM
2. อ่าน Function Call
3. Execute Tool
4. ส่ง Tool Result กลับให้ LLM
5. ให้ LLM ตัดสินใจว่าจะเรียก Tool เพิ่มหรือสรุปคำตอบ
6. เก็บ `trace` เพื่อใช้ Evaluation


In [ ]:
AGENT_INSTRUCTIONS = """
You are the ABC Beverage business AI agent.

Rules:
1. Exact transaction numbers must come from SQL/API tools. Never guess.
2. Company rules, policies, SOPs, or approval procedures must come from the RAG tool.
3. A question may require more than one tool.
4. If data is missing, say that it was not found.
5. Do not create a hold-order draft unless the user explicitly asks to create/prepare it.
6. Hold-order actions are DRAFT only and require human approval.
7. Keep numeric values faithful to tool results.
8. Mention the policy source when a recommendation depends on RAG.
"""


def run_agent(question, max_rounds=8, verbose=True):
    messages = [
        {
            "role": "system",
            "content": AGENT_INSTRUCTIONS
        },
        {
            "role": "user",
            "content": question
        }
    ]

    trace = []

    for round_number in range(1, max_rounds + 1):

        response = typhoon_client.chat.completions.create(
            model=TYPHOON_MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0.2,
            max_tokens=1500
        )

        assistant_message = response.choices[0].message

        messages.append(
            assistant_message.model_dump(
                exclude_none=True
            )
        )

        tool_calls = assistant_message.tool_calls or []

        # No tool call = final answer
        if not tool_calls:
            return {
                "answer": assistant_message.content or "",
                "trace": trace,
                "rounds": round_number,
                "provider": "typhoon"
            }

        for call in tool_calls:

            tool_name = call.function.name

            arguments = json.loads(
                call.function.arguments or "{}"
            )

            result = execute_tool(
                tool_name,
                arguments
            )

            trace.append({
                "round": round_number,
                "tool": tool_name,
                "arguments": arguments,
                "result": result
            })

            if verbose:
                print(
                    f"[Round {round_number}] TOOL:",
                    tool_name
                )
                print(
                    "Arguments:",
                    json.dumps(
                        arguments,
                        ensure_ascii=False
                    )
                )
                print(
                    "Result:",
                    json.dumps(
                        result,
                        ensure_ascii=False,
                        default=str
                    )[:1500]
                )
                print("-" * 100)

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(
                    result,
                    ensure_ascii=False,
                    default=str
                )
            })

    raise RuntimeError(
        f"Agent exceeded max_rounds={max_rounds}"
    )


## STEP 30 — Test 1: SQL Only

คำถามนี้ควรใช้ `get_overdue_invoices`


In [ ]:
test_sql = run_agent(
    "ตอนนี้ Invoice ที่ค้างชำระมากที่สุดมีรายการอะไรบ้าง เอา 5 รายการ"
)

print("\nFINAL ANSWER")
print(test_sql["answer"])
print("\nTOOLS:", [x["tool"] for x in test_sql["trace"]])


## STEP 31 — Test 2: RAG Only

คำถามนี้ควรใช้ `search_company_knowledge`


In [ ]:
test_rag = run_agent(
    "ตามนโยบายบริษัท ถ้าลูกค้าค้างเกิน 60 วันต้องทำอย่างไร"
)

print("\nFINAL ANSWER")
print(test_rag["answer"])
print("\nTOOLS:", [x["tool"] for x in test_rag["trace"]])


## STEP 32 — Test 3: SQL + RAG

นี่คือ Use Case สำคัญของ Agentic AI

คำถาม:

> ลูกค้ารายไหนควร Hold Order ตอนนี้ และเพราะอะไร?

Agent ควร:
- ใช้ SQL ดู Overdue
- ใช้ RAG อ่าน Policy
- รวม Evidence ก่อน Recommendation


In [ ]:
test_mixed = run_agent(
    "ลูกค้ารายไหนควร Hold Order ตอนนี้ ตามข้อมูลค้างชำระและนโยบายบริษัท? "
    "ยังไม่ต้องสร้าง Action"
)

print("\nFINAL ANSWER")
print(test_mixed["answer"])
print("\nTOOLS:", [x["tool"] for x in test_mixed["trace"]])


## STEP 33 — Test 4: Agent Action + Human-in-the-loop

คำถามนี้อนุญาตให้ Agent **สร้าง DRAFT Action**

แต่ยังไม่เปลี่ยนสถานะลูกค้าจริง


In [ ]:
test_action = run_agent(
    "วิเคราะห์ลูกค้าที่เข้าเงื่อนไข Hold Order ตามข้อมูลและ Policy "
    "แล้วสร้าง Hold Order DRAFT สำหรับรายที่ควรดำเนินการ"
)

print("\nFINAL ANSWER")
print(test_action["answer"])
print("\nTOOLS:", [x["tool"] for x in test_action["trace"]])


## STEP 34 — ตรวจ Action Queue

ผู้อนุมัติสามารถเปิดดูรายการ `DRAFT` ก่อนดำเนินการจริง


In [ ]:
actions = (
    supabase.table("agent_actions")
    .select("*")
    .order("created_at", desc=True)
    .limit(20)
    .execute()
)

pd.DataFrame(actions.data)


## STEP 35 — Open API Test

ใช้เมื่อมี API จริงแล้ว

ตัวอย่าง:

```python
get_external_api(
    resource="inventory",
    identifier="BEV001"
)
```

ก่อนต่อ Agent ให้ทดสอบ **API ตรง ๆ ก่อนเสมอ** ว่า JSON ที่ได้คือ Ground Truth


In [ ]:
# Uncomment after OPEN_API_BASE_URL is configured.
#
# api_result = get_external_api(
#     resource="inventory",
#     identifier="BEV001"
# )
#
# api_result


# PART 3 — Agent Evaluation

Agent ที่ตอบได้ยังไม่พอ ต้องวัดว่า:

- เลือก Tool ถูกไหม
- ใช้ Source ถูกไหม
- ตัวเลขตรงไหม
- เมื่อไม่มีข้อมูล กล้าบอกว่าไม่พบหรือไม่
- Action ถูกสร้างเป็น DRAFT หรือไม่

เราจะใช้ชีต `Agent_Test_Cases` จาก Excel เป็นชุดทดสอบ


## STEP 36 — Load Test Cases


In [ ]:
agent_test_cases = pd.read_excel(
    EXCEL_PATH,
    sheet_name="Agent_Test_Cases"
)

agent_test_cases


## STEP 37 — Route Mapping สำหรับ Evaluation

Dataset เดิมใช้คำว่า:
- SQL/API
- RAG
- SQL + RAG

เราจะแปลงเป็น Tool Family ที่คาดหวัง


In [ ]:
def tool_family(tool_name):
    if tool_name in {
        "get_overdue_invoices",
        "get_low_stock",
        "get_customer",
    }:
        return "SQL"

    if tool_name == "search_company_knowledge":
        return "RAG"

    if tool_name == "get_external_api":
        return "API"

    if tool_name == "create_hold_order_draft":
        return "ACTION"

    return "OTHER"


def expected_families(expected_route):
    route = str(expected_route).upper()

    families = set()

    if "SQL" in route:
        families.add("SQL")
    if "RAG" in route:
        families.add("RAG")
    if "API" in route and "SQL/API" not in route:
        families.add("API")

    # SQL/API means either structured source is acceptable for this lesson.
    if "SQL/API" in route:
        families.add("STRUCTURED")

    return families


## STEP 38 — Evaluate One Question


In [ ]:
def evaluate_one(question, expected_route):
    result = run_agent(
        question,
        verbose=False
    )

    tools_used = [x["tool"] for x in result["trace"]]
    families_used = {tool_family(t) for t in tools_used}
    expected = expected_families(expected_route)

    if "STRUCTURED" in expected:
        route_pass = bool(
            {"SQL", "API"} & families_used
        )
    else:
        route_pass = expected.issubset(families_used)

    return {
        "question": question,
        "expected_route": expected_route,
        "tools_used": tools_used,
        "route_pass": route_pass,
        "answer": result["answer"],
    }


In [ ]:
sample_eval = evaluate_one(
    "ถ้าลูกค้าค้างเกิน 60 วันต้องทำอะไร",
    "RAG"
)

sample_eval


## STEP 39 — Run Evaluation Dataset

หมายเหตุ: การรันทุก Test Case จะมี API cost เพราะ Agent และ Embedding ถูกเรียกหลายครั้ง


In [ ]:
def run_evaluation(test_df, max_tests=None):
    rows = []

    subset = test_df
    if max_tests is not None:
        subset = test_df.head(max_tests)

    for _, test in subset.iterrows():
        result = evaluate_one(
            question=test["question"],
            expected_route=test["expected_route"],
        )

        result["test_id"] = test["test_id"]
        result["expected_source"] = test["expected_source"]
        result["expected_answer"] = test["expected_answer"]

        rows.append(result)

        print(
            test["test_id"],
            "PASS" if result["route_pass"] else "FAIL",
            result["tools_used"]
        )

    return pd.DataFrame(rows)


In [ ]:
# เริ่มจาก 3 ข้อก่อนเพื่อประหยัด API cost
evaluation_results = run_evaluation(
    agent_test_cases,
    max_tests=3
)

evaluation_results


## STEP 40 — Route Accuracy


In [ ]:
route_accuracy = evaluation_results["route_pass"].mean()

print(f"Route Accuracy: {route_accuracy:.2%}")


# สรุป Architecture หลังจบ Lab

```text
                        USER
                          │
                          ▼
                    AI AGENT / LLM
                          │
          ┌───────────────┼────────────────┐
          │               │                │
          ▼               ▼                ▼
       SQL TOOL         RAG TOOL        OPEN API
          │               │                │
          ▼               ▼                ▼
      PostgreSQL       pgvector       ERP / CRM / HR
          │               │
          └───────┬───────┘
                  ▼
              ANALYSIS
                  │
                  ▼
          RECOMMENDATION
                  │
          Human Approval
                  │
                  ▼
             ACTION DRAFT
```

สิ่งที่ทำให้ Lab นี้เป็น Agentic AI ไม่ใช่แค่ RAG คือ Agent สามารถ:
- เลือก Tool
- เรียกหลาย Tool
- ตรวจ Evidence
- วนกลับไปหา Tool เพิ่ม
- สร้าง Action
- มี Human Approval
- มี Evaluation / Tool Trace
```
